# AI Dashcam: train YOLO11n trên Google Colab (CV12)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhhuypham/AIDashCam/blob/main/colab_train.ipynb)

Notebook này chỉ là **nút bấm**: toàn bộ logic nằm trong `dashcam_ml/` và `config.yaml`, code được lấy thẳng từ GitHub.

**Chuẩn bị một lần trên máy Windows** (mở terminal ở thư mục project):

```bash
tar -a -c -f bdd100k.zip -C data bdd100k
```

Tải `bdd100k.zip` lên Google Drive, vào thư mục `MyDrive/AIDashCam/`.
- Sửa code hoặc `config.yaml` trên máy thì chỉ cần `git push`, phiên Colab sau sẽ tự lấy bản mới.
- Đổi sang bản BDD100K đầy đủ = thay thư mục `data/bdd100k` rồi tạo lại `bdd100k.zip`.

**Mỗi phiên Colab:**
1. Mở notebook bằng nút **Open in Colab** ở trên.
2. Menu **Runtime → Change runtime type → T4 GPU**.
3. Xem lại ô **Thiết lập** bên dưới.
4. Menu **Runtime → Run all**.

Kết quả (`best.pt`, `last.pt`, biểu đồ) được ghi thẳng vào `MyDrive/AIDashCam/outputs/`, nên phiên bị ngắt cũng không mất.
Bị ngắt giữa chừng thì đặt `RESUME = True` rồi chạy lại từ đầu, quá trình train sẽ tiếp tục từ `last.pt`.

## 1. Thiết lập

In [ ]:
REPO_URL  = 'https://github.com/khanhhuypham/AIDashCam.git'
DRIVE_DIR = '/content/drive/MyDrive/AIDashCam'   # thư mục trên Drive chứa bdd100k.zip và outputs/
DATA_ZIP  = 'bdd100k.zip'

RESUME    = False   # True = train tiếp từ last.pt của lần trước (khi phiên bị ngắt)
BATCH     = 32      # GPU T4 16 GB với ảnh 640: 32 là vừa; báo lỗi hết bộ nhớ (CUDA OOM) thì giảm còn 16
WORKERS   = 2       # Colab chỉ có 2 nhân CPU

RUN_EVALUATE = True   # sau khi train: chấm điểm trọng số gốc COCO và best.pt trên cùng tập (CV11)
RUN_EXPORT   = True   # sau khi train: xuất best.pt sang Core ML (CV14)

WORK_DIR = '/content/AIDashCam'

## 2. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -lh {DRIVE_DIR}

## 4. Lấy code

In [ ]:
import os
if os.path.isdir(f'{WORK_DIR}/.git'):
    # bỏ các chỉnh sửa config.yaml của lần chạy trước rồi lấy bản mới nhất
    !git -C {WORK_DIR} checkout -- config.yaml && git -C {WORK_DIR} pull -q
else:
    !git clone -q {REPO_URL} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -1

## 5. Cài thư viện

Colab đã có sẵn PyTorch bản GPU. `coremltools` chỉ cài ở bước xuất Core ML.

In [ ]:
!grep -v coremltools requirements.txt > /tmp/requirements_train.txt
!pip install -q -r /tmp/requirements_train.txt

## 6. Dữ liệu

Copy **một** file zip về ổ của máy Colab rồi mới giải nén: đọc hàng chục nghìn ảnh trực tiếp từ Drive rất chậm.
Sau đó chuyển nhãn sang định dạng YOLO ngay trên Colab (`dataset.yaml` chứa đường dẫn tuyệt đối nên không dùng lại được bản tạo trên Windows).

In [ ]:
if not os.path.isfile('data/yolo_bdd100k/dataset.yaml'):
    !mkdir -p data && cp {DRIVE_DIR}/{DATA_ZIP} /content/{DATA_ZIP}
    !unzip -q -o /content/{DATA_ZIP} -d data && rm /content/{DATA_ZIP}
    !python -m dashcam_ml prepare-bdd100k
else:
    print('Dữ liệu đã sẵn sàng từ trước trong phiên này, bỏ qua.')
!head -5 data/yolo_bdd100k/dataset.yaml

## 7. Ghi kết quả thẳng vào Google Drive

`outputs/train` và `outputs/coreml` được trỏ sang `MyDrive/AIDashCam/outputs/`, nên `best.pt` / `last.pt` nằm trên Drive ngay khi được lưu.
`outputs/eval` để trên máy Colab vì bước đánh giá copy ảnh vào đó; chỉ file báo cáo được chép lên Drive.

In [ ]:
for sub in ['train', 'coreml']:
    !mkdir -p outputs {DRIVE_DIR}/outputs/{sub}
    !rm -rf outputs/{sub} && ln -s {DRIVE_DIR}/outputs/{sub} outputs/{sub}
!ls -la outputs

## 8. Chỉnh `config.yaml` cho máy Colab

Chỉ sửa bản copy trên máy Colab; `config.yaml` trên máy Windows không đổi.

In [ ]:
import re, pathlib, yaml
CFG = pathlib.Path('config.yaml')

def set_cfg(section, key, value):
    # Đổi giá trị section.key trong config.yaml, giữ nguyên chú thích.
    text = CFG.read_text(encoding='utf-8')
    block = re.search(rf'^{section}:\n(?:(?:[ \t].*)?\n)*', text, flags=re.M).group(0)
    new_block, n = re.subn(rf'^(    {key}:)[ \t]*[^#\n]*?([ \t]*(?:#.*)?)$', rf'\g<1> {value}\g<2>', block, count=1, flags=re.M)
    assert n == 1, f'không tìm thấy {section}.{key} trong config.yaml'
    CFG.write_text(text.replace(block, new_block, 1), encoding='utf-8')
    print(f'{section}.{key} = {value}')

RUN_NAME = yaml.safe_load(CFG.read_text(encoding='utf-8'))['train']['run_name']
BEST = f'outputs/train/{RUN_NAME}/weights/best.pt'
LAST = f'outputs/train/{RUN_NAME}/weights/last.pt'

set_cfg('train', 'batch', BATCH)
set_cfg('train', 'workers', WORKERS)
set_cfg('evaluate', 'batch', BATCH)
if RESUME:
    assert os.path.isfile(LAST), f'chưa có {LAST} trên Drive, không train tiếp được'
    set_cfg('train', 'model', LAST)
    set_cfg('train', 'resume', 'true')

## 9. Train (CV12)

In [ ]:
!python -m dashcam_ml train
!ls -lh outputs/train/{RUN_NAME}/weights

## 10. Đánh giá (CV11): trọng số gốc COCO so với `best.pt`

Kết quả: `MyDrive/AIDashCam/outputs/eval/evaluation_report.json`.

In [ ]:
if RUN_EVALUATE:
    text = CFG.read_text(encoding='utf-8')
    CFG.write_text(text.replace(f'# - {BEST}', f'- {BEST}'), encoding='utf-8')
    !python -m dashcam_ml evaluate
    !mkdir -p {DRIVE_DIR}/outputs/eval
    !cp outputs/eval/evaluation_report.json {DRIVE_DIR}/outputs/eval/ && cp -r outputs/eval/runs {DRIVE_DIR}/outputs/eval/

## 11. Xuất Core ML (CV14)

Colab chạy Linux nên xuất được (Windows thì không). Các file `.mlpackage` nằm ở `MyDrive/AIDashCam/outputs/coreml/`.
Bước `verify-coreml` vẫn phải chạy trên Mac.

In [ ]:
if RUN_EXPORT:
    !pip install -q "coremltools>=8.0"
    set_cfg('coreml_export', 'weights', BEST)
    !python -m dashcam_ml export-coreml
    !python -m dashcam_ml export-ios-config
    !ls -lh outputs/coreml